# How to asynchronously run and profile a CUDA program

## How to link cub and nvtx library

In [15]:
import os
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]


I found even my `nvcc` version is 12.4. Some header files are still not included. So it's better to install the cccl package via conda.
```bash
conda install -c conda-forge cccl
conda install -c conda-forge nvtx



ln -s $CONDA_PREFIX/include/cccl/cub cub
ln -s $CONDA_PREFIX/include/cccl/thrust thrust
ln -s $CONDA_PREFIX/include/cccl/cuda cuda
ln -s $CONDA_PREFIX/include/cccl/nv nv
```




When we compile the CUDA program with `nvcc`, 
error:
```bash
nvcc  --extended-lambda -o /tmp/a.out Sources/nvtx.cpp -x cu -arch=native 
```
correct:
```bash
nvcc -I$CONDA_PREFIX/include --extended-lambda -o /tmp/a.out Sources/nvtx.cpp -x cu -arch=native 
```


**overlap**: to make two or more things happen at the same time.

In cuda, asynchronous execution means that the CPU can continue executing other code while the GPU is busy processing data. This allows for better utilization of both the CPU and GPU, leading to improved performance in applications that require heavy computation.


`cub::DeviceTransform::Transform` : A function from the CUB library that performs transformations on device data asynchronously.

API
```cpp
cub::DeviceTransform::Transform(input_iterator, output_iterator, num_items, op, stream);

```



In [2]:
%%writefile Sources/nvtx.cpp

#include <fstream>
#include <thrust/fill.h>
#include <thrust/device_vector.h>
#include <thrust/host_vector.h>
#include <cuda/std/mdspan>
#include <nvtx3.hpp>
#include <cub/device/device_transform.cuh>


void simulate(int width, int height, const thrust::device_vector<float> &in, thrust::device_vector<float> &out) {
    cuda::std::mdspan temp_in(thrust::raw_pointer_cast(in.data()), height, width);

    cub::DeviceTransform::Transform( 
        thrust::make_counting_iterator(0),
        out.begin(),
        width * height,
        [=] __device__ __host__ (int id){
            int row = id / width;
            int column = id % width;

            if (row > 0 && column > 0 && row < height - 1 && column < width - 1) {
                float d2tdx2 =
                    temp_in(row, column - 1) - 2 * temp_in(row, column) + temp_in(row, column + 1);
                float d2tdy2 =
                    temp_in(row - 1, column) - 2 * temp_in(row, column) + temp_in(row + 1, column);

                return temp_in(row, column) + 0.2f * (d2tdx2 + d2tdy2);
            } else {
                return temp_in(row, column);
            }
        }
    );
}

thrust::device_vector<float> init(int width, int height) {
    thrust::device_vector<float> d_prev(width * height, 15.0f);
    thrust::fill_n(d_prev.begin(), width, 90.0f);
    thrust::fill_n(d_prev.begin() + width * (height - 1), width, 90.0f);
    return d_prev;
    
}

template <class ContainerT>
void store(int step, int height, int width, ContainerT &data)
{
    std::ofstream file("/tmp/heat_" + std::to_string(step) + ".bin", std::ios::binary);
    file.write(reinterpret_cast<const char*>(&height), sizeof(int));
    file.write(reinterpret_cast<const char*>(&width), sizeof(int));
    file.write(reinterpret_cast<const char *>(thrust::raw_pointer_cast(data.data())), height * width * sizeof(float));
}

int main(){
    int height = 2048;
    int width = 8192;
    thrust::device_vector<float> d_prev = init(width, height);
    thrust::device_vector<float> d_next(height * width);
    thrust::host_vector<float> h_prev(height * width);
    const int compute_steps = 750;
    const int write_steps = 3;
    for (int write_step = 0; write_step < write_steps; write_step++)
    {
        nvtx3::scoped_range r{std::string("write step ") + std::to_string(write_step)};

        {
            nvtx3::scoped_range r{"copy"};
            thrust::copy(d_prev.begin(), d_prev.end(), h_prev.begin());
        }

        {
            nvtx3::scoped_range r{"compute"};
            for (int compute_step = 0; compute_step < compute_steps; compute_step++)
            {
                // Even though the simulate function is called asynchronously, `swap` will not fall into the data race. Because eveything is serialized in a single default stream.
                simulate(width, height, d_prev, d_next);
                d_prev.swap(d_next);
            }
        }

        {
            nvtx3::scoped_range r{"write"};
            store(write_step, height, width, h_prev);
        }

        {
            nvtx3::scoped_range r{"wait"};
            cudaDeviceSynchronize();
        }
    }
}



Overwriting Sources/nvtx.cpp


In [3]:
!nvcc -I $CONDA_PREFIX/include --extended-lambda -o /tmp/a.out Sources/nvtx.cpp -x cu -arch=native # build executable
!nsys profile --force-overwrite true -o nvtx /tmp/a.out # run and profile executable

Generating '/tmp/nsys-report-a303.qdstrm'
[1/1] [========================100%] nvtx.nsys-rep
Generated:
    /home/huhu/LLM_NOTEBOOK/docs/1_cuda_from_scratch/nvtx.nsys-rep



<img src="https://github.com/rhu2xx/picx-images-hosting/raw/master/20260202/image.8adrsvcymy.webp" style="width:90%;">

Right now, the simulator:
1. synchronously copies data from GPU to CPU.
2. overlaps computation and I/O to some extent.
3. Waits for the copy to finish.


End till now, the overlap only happens between computation and I/O. The data copy is still synchronous. We use `cudaMemcpyAsync` to replace `thrust::copy` for asynchronous copy.
But if we only use `cudaMemcpyAsync`, the overlap will not happen. Because the default stream is used. All operations in the default stream are serialized. To achieve true overlap, we need to use multiple streams.

API
```cpp
cudaMemcpyAsync(void* dst, const void* src, size_t count_byte, cudaMemcpyKind kind, cudaStream_t stream);
```


In [9]:
%%writefile Sources/async-copy.cpp

#include <fstream>
#include <thrust/fill.h>
#include <thrust/device_vector.h>
#include <thrust/host_vector.h>
#include <cuda/std/mdspan>
#include "nvtx3.hpp"
#include <cub/device/device_transform.cuh>


void simulate(int width, int height, const thrust::device_vector<float> &in, thrust::device_vector<float> &out, cudaStream_t stream) {
    cuda::std::mdspan temp_in(thrust::raw_pointer_cast(in.data()), height, width);

    cub::DeviceTransform::Transform( 
        thrust::make_counting_iterator(0),
        out.begin(),
        width * height,
        [=] __device__ __host__ (int id){
            int row = id / width;
            int column = id % width;

            if (row > 0 && column > 0 && row < height - 1 && column < width - 1) {
                float d2tdx2 =
                    temp_in(row, column - 1) - 2 * temp_in(row, column) + temp_in(row, column + 1);
                float d2tdy2 =
                    temp_in(row - 1, column) - 2 * temp_in(row, column) + temp_in(row + 1, column);

                return temp_in(row, column) + 0.2f * (d2tdx2 + d2tdy2);
            } else {
                return temp_in(row, column);
            }
        },
        stream);
}

thrust::device_vector<float> init(int width, int height) {
    thrust::device_vector<float> d_prev(width * height, 15.0f);
    thrust::fill_n(d_prev.begin(), width, 90.0f);
    thrust::fill_n(d_prev.begin() + width * (height - 1), width, 90.0f);
    return d_prev;
    
}

template <class ContainerT>
void store(int step, int height, int width, ContainerT &data)
{
    std::ofstream file("/tmp/heat_" + std::to_string(step) + ".bin", std::ios::binary);
    file.write(reinterpret_cast<const char*>(&height), sizeof(int));
    file.write(reinterpret_cast<const char*>(&width), sizeof(int));
    file.write(reinterpret_cast<const char *>(thrust::raw_pointer_cast(data.data())), height * width * sizeof(float));
}

int main(){
    int height = 2048;
    int width = 8192;

    cudaStream_t compute_stream;
    cudaStreamCreate(&compute_stream);

    cudaStream_t copy_stream;
    cudaStreamCreate(&copy_stream);

    thrust::device_vector<float> d_prev = init(width, height);
    thrust::device_vector<float> d_buffer(height * width);
    thrust::device_vector<float> d_next(height * width);
    thrust::host_vector<float> h_prev(height * width);
    const int compute_steps = 750;
    const int write_steps = 3;
    for (int write_step = 0; write_step < write_steps; write_step++)
    {
        nvtx3::scoped_range r{std::string("write step ") + std::to_string(write_step)};

        {
            nvtx3::scoped_range r{"copy"};
            cudaMemcpy(thrust::raw_pointer_cast(d_buffer.data()), thrust::raw_pointer_cast(d_prev.data()), d_prev.size() * sizeof(float), cudaMemcpyDeviceToDevice);
            cudaMemcpyAsync(thrust::raw_pointer_cast(h_prev.data()), thrust::raw_pointer_cast(d_buffer.data()), d_prev.size() * sizeof(float), cudaMemcpyDeviceToHost, copy_stream);

        }

        {
            nvtx3::scoped_range r{"compute"};
            for (int compute_step = 0; compute_step < compute_steps; compute_step++)
            {
                // Read-Write conflict will not happen. 
                simulate(width, height, d_prev, d_next, compute_stream);
                d_prev.swap(d_next);
            }
        }

        {
            nvtx3::scoped_range r{"write"};
            cudaStreamSynchronize(copy_stream);
            store(write_step, height, width, h_prev);
        }

        {
            nvtx3::scoped_range r{"wait"};
            cudaStreamSynchronize(compute_stream);
        }
    }
    cudaStreamDestroy(compute_stream);
    cudaStreamDestroy(copy_stream);
}



Overwriting Sources/async-copy.cpp


In [10]:
!nvcc -I$CONDA_PREFIX/include --extended-lambda -o /tmp/a.out Sources/async-copy.cpp -x cu -arch=native # build executable
!nsys profile --force-overwrite true -o nvtx-async /tmp/a.out # run and profile executable

Generating '/tmp/nsys-report-8eeb.qdstrm'
[1/1] [========================100%] nvtx-async.nsys-rep
Generated:
    /home/huhu/LLM_NOTEBOOK/docs/1_cuda_from_scratch/nvtx-async.nsys-rep
